In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE     = next(p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / "DECISIONS.md").exists())
PROC     = BASE / "data/processed"
TABS     = BASE / "outputs/tables"

print("══ CHECKLIST DE CONSISTÊNCIA DO PIPELINE ══════════════")
resultados_check = {}

In [ ]:
print("\n── C1: Arquivos de dados processados ───────────────────")

arquivos_esperados = {
    "pnadc_2023_anual.csv":                    (1_800_000, 36),  
    "horas_por_setor_2023_corrigido.csv":      (11, 9),
    "pib_setorial_2012_2023.csv":              (12, 14),        
    "base_analitica_setorial_2023.csv":        (10, 17),
    "faixas_por_setor_2023.csv":               (11, 9),
}

c1_ok = True
for nome, (min_linhas, min_colunas) in arquivos_esperados.items():
    path = PROC / nome
    if not path.exists():
        print(f"  ✗ FALTANDO: {nome}")
        c1_ok = False
        continue
    df = pd.read_csv(path)
    ok_linhas  = len(df) >= min_linhas
    ok_colunas = len(df.columns) >= min_colunas
    status = "✓" if ok_linhas and ok_colunas else "✗"
    print(f"  {status} {nome}: {len(df)} linhas × {len(df.columns)} colunas")
    if not (ok_linhas and ok_colunas):
        c1_ok = False

resultados_check["C1_arquivos_existem"] = c1_ok

In [ ]:
print("\n── C2: Consistência da definição de ΔH ─────────────────")

horas_faixa   = np.array([41, 42, 43, 44])
DELTA_H_C_REF = ((40 - horas_faixa) / horas_faixa).mean()

df_base = pd.read_csv(PROC / "base_analitica_setorial_2023.csv")

if "delta_h_pct" in df_base.columns:
    delta_h_no_arquivo = df_base["delta_h_pct"].iloc[0] / 100
    consistente = abs(delta_h_no_arquivo - DELTA_H_C_REF) < 1e-6
    print(f"  ΔH referência (Formulação C): {DELTA_H_C_REF*100:.6f}%")
    print(f"  ΔH no arquivo base_analitica: {delta_h_no_arquivo*100:.6f}%")
    print(f"  Consistente: {'✓' if consistente else '✗ DIVERGÊNCIA'}")
    resultados_check["C2_delta_h_consistente"] = consistente
else:
    print("  ✗ Coluna delta_h_pct não encontrada em base_analitica")
    resultados_check["C2_delta_h_consistente"] = False

In [ ]:
print("\n── C3: Consistência da base de dados ───────────────────")

df_horas = pd.read_csv(PROC / "horas_por_setor_2023_corrigido.csv", index_col=0)
df_pib   = pd.read_csv(PROC / "pib_setorial_2012_2023.csv")

pib_2023     = df_pib[df_pib["ano"] == 2023].squeeze()
pib_total_rs = pib_2023["pib_total"] * 1e6

correspondencia = [
    ("Agropecuária",          "agropecuaria",          True),
    ("Indústria geral",       "ind_transformacao",     True),
    ("Construção",            "construcao",            True),
    ("Comércio e rep.",       "comercio",              True),
    ("Transp. e armaz.",      "transporte",            True),
    ("Inf. e serv. prof.",    "info_comunicacao",      True),
    ("Adm. públ. e educação", "adm_publica_saude_educ",True),
]

delta_vab_recalc = 0
for setor, col, _ in correspondencia:
    if setor not in df_horas.index:
        continue
    vab_rs  = pib_2023[col] * 1e6
    parcela = df_horas.loc[setor, "pct_41_44h"] / 100.0
    delta_vab_recalc += vab_rs * parcela * DELTA_H_C_REF

delta_pib_recalc = delta_vab_recalc / pib_total_rs * 100

delta_pib_arquivo = df_base[df_base["incluir"] == True]["delta_vab_r$_bi"].sum() * 1e9 / pib_total_rs * 100

diff = abs(delta_pib_recalc - delta_pib_arquivo)
consistente_c3 = diff < 0.001

print(f"  ΔPIB recalculado do zero:    {delta_pib_recalc:.6f}%")
print(f"  ΔPIB no arquivo base:        {delta_pib_arquivo:.6f}%")
print(f"  Diferença:                   {diff:.6f} pp")
print(f"  Consistente (< 0,001 pp): {'✓' if consistente_c3 else '✗ DIVERGÊNCIA'}")
resultados_check["C3_base_dados_consistente"] = consistente_c3

In [ ]:
print("\n── C4: Tabelas de output ────────────────────────────────")

tabelas_esperadas = [
    "cenario_base_setorial.csv",
    "matriz_hipoteses_completa.csv",
    "normalizacao_setorial.csv",
]

c4_ok = True
for nome in tabelas_esperadas:
    path = TABS / nome
    exists = path.exists()
    print(f"  {'✓' if exists else '✗'} {nome}")
    if not exists:
        c4_ok = False

if (TABS / "matriz_hipoteses_completa.csv").exists():
    df_mat = pd.read_csv(TABS / "matriz_hipoteses_completa.csv")
    valor_bl_0 = df_mat[
        (df_mat["faixa"] == "41–44h") &
        (df_mat["ganho_p"] == "0%")
    ]["delta_pib_incl"].values[0]

    esperado = delta_pib_recalc
    diff_mat = abs(valor_bl_0 - esperado)
    ok_mat   = diff_mat < 0.001
    print(f"\n  Verificação cruzada matriz × recálculo:")
    print(f"    Baseline 41–44h, 0% na matriz: {valor_bl_0:.6f}%")
    print(f"    Recálculo independente:         {esperado:.6f}%")
    print(f"    Diferença: {diff_mat:.6f} pp — {'✓' if ok_mat else '✗ DIVERGÊNCIA'}")
    if not ok_mat:
        c4_ok = False

resultados_check["C4_outputs_consistentes"] = c4_ok

In [ ]:
print("\n── C5: Reprodutibilidade ────────────────────────────────")

deps_ok = all([
    (PROC / "pnadc_2023_anual.csv").exists(),
    (PROC / "horas_por_setor_2023_corrigido.csv").exists(),
    (PROC / "pib_setorial_2012_2023.csv").exists(),
])
print(f"  Todos os inputs necessários presentes: {'✓' if deps_ok else '✗'}")

soma_ok = (
    df_horas["pct_abaixo_40"] + df_horas["pct_40h_exatas"] +
    df_horas["pct_41_44h"] + df_horas["pct_acima_44h"]
).between(99, 101).all()
print(f"  Faixas somam ~100% em todos os setores: {'✓' if soma_ok else '✗'}")

resultados_check["C5_reprodutivel"] = deps_ok and soma_ok

In [ ]:
print("\n══ RESULTADO FINAL ══════════════════════════════════════")
todos_ok = all(resultados_check.values())

for chave, resultado in resultados_check.items():
    print(f"  {'✓' if resultado else '✗'} {chave}")

print()
if todos_ok:
    print("  PIPELINE CONSISTENTE — pronto para avaliação de readiness")
else:
    print("  PIPELINE COM PROBLEMAS — resolver antes de avançar")
    falhas = [k for k, v in resultados_check.items() if not v]
    print(f"  Falhas: {falhas}")